# Working with agent harnesses

Let's follow a CSV bug through a harness, an MCP tool, project memory and
repeated model requests. The tool calls below are chosen explicitly so that
we can inspect the messages. The notebook runs on a CPU without API keys.

## 1. What the harness does

Task: fix CSV quoting while keeping the public interface, then inspect the
regression and the diff. An instruction file can describe these requirements.
A skill can supply the procedure. The harness supplies file access, tool
execution, permission checks, context assembly and session state.

| Harness / setup | What the product emphasizes | Technical details |
|---|---|---|
| [Codex](https://openai.com/codex/) | Delegating engineering work: parallel agents, cloud tasks, code review and plugins. Repository instructions include AGENTS.md. | [Instructions](https://developers.openai.com/codex/guides/agents-md/) |
| [Claude Code](https://claude.com/product/claude-code) | Working inside a repository from the terminal or IDE: search the codebase, edit files, run tests, turn issues into PRs. | [Overview](https://code.claude.com/docs/en/overview) |
| [OpenCode](https://opencode.ai/) | An open-source coding agent with a choice of providers and local models, parallel sessions, and terminal/desktop/editor interfaces. Tool permissions are configurable. | [Permissions](https://opencode.ai/v2/docs/permissions) |
| [Hermes](https://hermes-agent.nousresearch.com/) | An assistant that carries work across sessions: persistent memory, reusable/generated skills, messaging integrations and scheduled tasks. | [Skills](https://hermes-agent.nousresearch.com/docs/guides/work-with-skills) |
| [DeepSeek Harness](https://www.deepseek.com/en/harness/) | A plugin-based harness: extend tools, skills and UI; Creator Mode helps build plugins. Includes a developer trace view. The site labels it a preview. | [Repository](https://github.com/deepseek-ai/deepseek-harness) |
| [claude-brainlab](https://github.com/brain-lab-research/claude-brainlab) | A research workflow toolkit layered on Claude Code: skills, commands, specialist agents, hooks, literature/experiment/writing workflows and memory integration. | [Skills](https://github.com/brain-lab-research/claude-brainlab/blob/main/SKILLS.md) |

Let's open the official product pages and compare what they emphasize.
These are the publishers' descriptions, not a ranking or a claim that the
listed features are exclusive to one harness. The final entry, claude-brainlab,
packages a research setup on top of Claude Code.

An application UI can display the proposed diff and request approval.
[CopilotKit](https://docs.copilotkit.ai/concepts/architecture) connects UI and
agent backends using AG-UI. MCP connects the backend to tool servers.

```text
Application UI ↔ AG-UI ↔ agent backend ↔ MCP ↔ tool server
```

In [1]:
# @title Local tool helpers
import importlib.util
from pathlib import Path
from tempfile import TemporaryDirectory
runtime_files = TemporaryDirectory(prefix="agent-seminar-")
runtime_path = Path(runtime_files.name)
(runtime_path / 'mcp_server.py').write_text('"""Minimal teaching subset of MCP 2025-06-18 over newline-delimited stdio.\n\nNo model, files, network, or external dependencies. It exposes one CSV tool.\nThis is a protocol example, not a general-purpose MCP server implementation.\n"""\nimport csv\nimport io\nimport json\nimport sys\n\nVERSION = "2025-06-18"\nTOOL = {\n    "name": "inspect_csv",\n    "description": "Parse CSV text and return its columns, row count and first row.",\n    "inputSchema": {\n        "type": "object",\n        "properties": {"csv_text": {"type": "string"}},\n        "required": ["csv_text"],\n        "additionalProperties": False,\n    },\n}\n\n\ndef inspect_csv(csv_text):\n    rows = list(csv.reader(io.StringIO(csv_text), strict=True))\n    if not rows:\n        raise ValueError("CSV is empty.")\n    columns, *data = rows\n    if len(set(columns)) != len(columns):\n        raise ValueError("Column names must be unique.")\n    for number, row in enumerate(data, 2):\n        if len(row) != len(columns):\n            raise ValueError(\n                f"Row {number} has {len(row)} fields; expected {len(columns)}. "\n                "A comma inside a field must be quoted."\n            )\n    return {\n        "columns": columns,\n        "row_count": len(data),\n        "first_row": dict(zip(columns, data[0])) if data else {},\n    }\n\n\ndef serve():\n    negotiated = ready = False\n    for line in sys.stdin:\n        try:\n            request = json.loads(line)\n        except json.JSONDecodeError:\n            print(json.dumps({"jsonrpc": "2.0", "id": None,\n                              "error": {"code": -32700, "message": "Parse error"}}), flush=True)\n            continue\n        method = request.get("method")\n        if "id" not in request:\n            if method == "notifications/initialized" and negotiated:\n                ready = True\n            continue\n        response = {"jsonrpc": "2.0", "id": request["id"]}\n        if method == "initialize":\n            negotiated = True\n            result = {"protocolVersion": VERSION, "capabilities": {"tools": {}},\n                      "serverInfo": {"name": "csv-lab", "version": "1.0.0"}}\n        elif method == "ping":\n            result = {}\n        elif not ready:\n            response["error"] = {"code": -32002, "message": "Initialization is not complete"}\n        elif method == "tools/list":\n            result = {"tools": [TOOL]}\n        elif method == "tools/call":\n            params = request.get("params", {})\n            args = params.get("arguments", {})\n            if (params.get("name") != TOOL["name"] or not isinstance(args, dict)\n                    or set(args) != {"csv_text"} or not isinstance(args["csv_text"], str)):\n                response["error"] = {"code": -32602, "message": "Unknown tool or invalid arguments"}\n            else:\n                try:\n                    body = json.dumps(inspect_csv(args["csv_text"]), ensure_ascii=False)\n                    result = {"content": [{"type": "text", "text": body}], "isError": False}\n                except (ValueError, csv.Error) as error:\n                    result = {"content": [{"type": "text", "text": str(error)}], "isError": True}\n        else:\n            response["error"] = {"code": -32601, "message": "Method not found"}\n        if "error" not in response:\n            response["result"] = result\n        print(json.dumps(response, ensure_ascii=False), flush=True)\n\n\nif __name__ == "__main__":\n    sys.stdin.reconfigure(encoding="utf-8")\n    sys.stdout.reconfigure(encoding="utf-8")\n    serve()\n', encoding='utf-8')
(runtime_path / 'demo.py').write_text('"""Run the local CSV/MCP exchange and an explicit prefix-cache accounting example."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nGOOD_CSV = \'name,amount\\n"Doe, Jane",120\\n\'\nBAD_CSV = \'name,amount\\nDoe, Jane,120\\n\'\n\n\ndef mcp_exchange(wire=False):\n    server = Path(__file__).with_name("mcp_server.py")\n    process = subprocess.Popen(\n        [sys.executable, "-u", str(server)], stdin=subprocess.PIPE,\n        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, encoding="utf-8",\n    )\n    transcript = []\n\n    def send(request):\n        transcript.append({"direction": "client → server", "message": request})\n        process.stdin.write(json.dumps(request) + "\\n")\n        process.stdin.flush()\n        if "id" not in request:\n            return None\n        line = process.stdout.readline()\n        if not line:\n            raise RuntimeError("CSV server stopped before returning a response")\n        response = json.loads(line)\n        assert response["id"] == request["id"]\n        transcript.append({"direction": "server → client", "message": response})\n        return response\n\n    def request(number, method, params=None):\n        return send({"jsonrpc": "2.0", "id": number, "method": method,\n                     "params": {} if params is None else params})\n\n    try:\n        hello = request(1, "initialize", {\n            "protocolVersion": "2025-06-18", "capabilities": {},\n            "clientInfo": {"name": "seminar-client", "version": "1.0.0"},\n        })\n        assert hello["result"]["protocolVersion"] == "2025-06-18"\n        send({"jsonrpc": "2.0", "method": "notifications/initialized"})\n        tools = request(2, "tools/list")["result"]["tools"]\n        assert tools[0]["name"] == "inspect_csv"\n        calls = []\n        for number, source in [(3, BAD_CSV), (4, GOOD_CSV)]:\n            result = request(number, "tools/call", {\n                "name": "inspect_csv", "arguments": {"csv_text": source},\n            })["result"]\n            calls.append(result)\n        assert calls[0]["isError"] is True\n        parsed = json.loads(calls[1]["content"][0]["text"])\n        assert calls[1]["isError"] is False\n        assert parsed == {"columns": ["name", "amount"], "row_count": 1,\n                          "first_row": {"name": "Doe, Jane", "amount": "120"}}\n        invalid = request(5, "tools/call", {\n            "name": "inspect_csv", "arguments": {"csv_text": 42},\n        })\n        assert invalid["error"]["code"] == -32602\n    finally:\n        process.stdin.close()\n        try:\n            process.wait(timeout=5)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait()\n        diagnostics = process.stderr.read()\n        process.stdout.close()\n        process.stderr.close()\n    if process.returncode:\n        raise RuntimeError(diagnostics)\n    print("Local MCP stdio exchange; calls chosen by the demo client, no LLM.")\n    print("Discovered tool:", tools[0]["name"])\n    print("Bad CSV:", calls[0]["content"][0]["text"])\n    print("Fixed CSV:", json.dumps(parsed, ensure_ascii=False))\n    print("Wrong argument type: JSON-RPC error -32602 (no tool execution).")\n    if wire:\n        for event in transcript:\n            print("\\n" + event["direction"])\n            print(json.dumps(event["message"], indent=2, ensure_ascii=False))\n    return transcript\n\n\ndef cache_accounting():\n    # Assigned token lengths, not tokenizer output, measured latency or API usage.\n    # Turn 2 appends 200 assistant + 8,000 tool tokens. Turn 3 appends 200 + 2,000.\n    prompts = [4000, 12200, 14400]\n    # A hit reuses exactly the preceding prompt. No caching of its generated output\n    # is assumed. Real providers may cache more or less under their own policies.\n    cached_prefix = [0, prompts[0], prompts[1]]\n    suffixes = [full - hit for full, hit in zip(prompts, cached_prefix)]\n    assert sum(prompts) == 30600\n    assert sum(suffixes) == 14400\n    print("\\nAssigned token lengths; full hits on the preceding prompt prefix:")\n    print("Turn | Input | Cached prefix | New prefill")\n    for number, (full, cached, suffix) in enumerate(zip(prompts, cached_prefix, suffixes), 1):\n        print(f"{number:4} | {full:5,} | {cached:13,} | {suffix:11,}")\n    print(f"Without reuse: {sum(prompts):,} input token positions processed across requests.")\n    print(f"With the stated hits: {sum(suffixes):,} uncached positions + {sum(cached_prefix):,} cached positions.")\n    print("The 8,000-token tool result is first processed on turn 2; it is reusable on turn 3.")\n    print("If an early instruction changes, the matched prefix may shrink; do not count that as a hit.")\n    print("Token counts do not determine a FLOP, latency or price ratio.")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--wire", action="store_true", help="Print every actual JSON-RPC message")\n    args = parser.parse_args()\n    mcp_exchange(args.wire)\n    cache_accounting()\n', encoding='utf-8')
spec = importlib.util.spec_from_file_location("local_mcp_demo", runtime_path / "demo.py")
local_demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(local_demo)
print("Local CSV tool ready. CPU only; no model or network calls.")


Local CSV tool ready. CPU only; no model or network calls.


In [2]:
import csv
import io
import json
from html import escape
from IPython.display import HTML, display

name = "Doe, Jane"
bad_csv = f"name,amount\n{name},120\n"
stream = io.StringIO(newline="")
writer = csv.writer(stream, lineterminator="\n")
writer.writerow(["name", "amount"])
writer.writerow([name, "120"])
good_csv = stream.getvalue()
bad_fields = list(csv.reader(io.StringIO(bad_csv)))[1]
good_fields = list(csv.reader(io.StringIO(good_csv)))[1]
assert len(bad_fields) == 3
assert good_fields == [name, "120"]
cards = []
for title, source, fields in [
    ("Join with commas", bad_csv, bad_fields),
    ("Use csv.writer", good_csv, good_fields),
]:
    cards.append(
        f'<div style="flex:1;min-width:260px;border:1px solid #bcc9d6;padding:16px">'
        f'<b>{escape(title)}</b><pre>{escape(source)}</pre>'
        f'<p>Parsed fields: {len(fields)}</p><pre>{escape(json.dumps(fields))}</pre></div>'
    )
display(HTML('<div style="display:flex;gap:16px;flex-wrap:wrap">' + "".join(cards) + "</div>"))

## 2. Inspect MCP requests and results

Let's start a local process exposing `inspect_csv`. It receives real JSON-RPC
messages over stdio. The protocol version is pinned to `2025-06-18`.

```text
initialize → response → notifications/initialized
tools/list → tools/call → result
```

The model's tool-call message and the MCP request are different messages. The
host connects them. Here the client selects the calls directly, without a model.

In [3]:
transcript = local_demo.mcp_exchange()
sent = {event["message"]["id"]: event["message"] for event in transcript
        if event["direction"] == "client → server" and "id" in event["message"]}
received = {event["message"]["id"]: event["message"] for event in transcript
            if event["direction"] == "server → client"}
tool = received[2]["result"]["tools"][0]
print("Tool schema returned by tools/list:")
print(json.dumps(tool, indent=2))

Local MCP stdio exchange; calls chosen by the demo client, no LLM.
Discovered tool: inspect_csv
Bad CSV: Row 2 has 3 fields; expected 2. A comma inside a field must be quoted.
Fixed CSV: {"columns": ["name", "amount"], "row_count": 1, "first_row": {"name": "Doe, Jane", "amount": "120"}}
Wrong argument type: JSON-RPC error -32602 (no tool execution).
Tool schema returned by tools/list:
{
  "name": "inspect_csv",
  "description": "Parse CSV text and return its columns, row count and first row.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "csv_text": {
        "type": "string"
      }
    },
    "required": [
      "csv_text"
    ],
    "additionalProperties": false
  }
}


In [4]:
print("Client → server, malformed CSV in a valid tool call:")
print(json.dumps(sent[3], indent=2))
print("\nServer → client, error returned by the executed tool:")
print(json.dumps(received[3], indent=2))
print("\nCorrected call, decoded text payload:")
parsed_result = json.loads(received[4]["result"]["content"][0]["text"])
print(json.dumps(parsed_result, indent=2))
assert received[3]["result"]["isError"] is True
assert received[4]["result"]["isError"] is False
assert parsed_result["first_row"]["name"] == name

Client → server, malformed CSV in a valid tool call:
{
  "jsonrpc": "2.0",
  "id": 3,
  "method": "tools/call",
  "params": {
    "name": "inspect_csv",
    "arguments": {
      "csv_text": "name,amount\nDoe, Jane,120\n"
    }
  }
}

Server → client, error returned by the executed tool:
{
  "jsonrpc": "2.0",
  "id": 3,
  "result": {
    "content": [
      {
        "type": "text",
        "text": "Row 2 has 3 fields; expected 2. A comma inside a field must be quoted."
      }
    ],
    "isError": true
  }
}

Corrected call, decoded text payload:
{
  "columns": [
    "name",
    "amount"
  ],
  "row_count": 1,
  "first_row": {
    "name": "Doe, Jane",
    "amount": "120"
  }
}


What changes if `csv_text` is a number? The arguments violate the schema, so
the tool does not execute. The malformed CSV above has a valid string argument
and produces an execution error inside `result`.

Sources: [MCP tools](https://modelcontextprotocol.io/specification/2025-06-18/server/tools),
[lifecycle](https://modelcontextprotocol.io/specification/2025-06-18/basic/lifecycle).

In [5]:
print("Wrong argument type, csv_text=42:")
print(json.dumps(received[5], indent=2))
assert received[5]["error"]["code"] == -32602

Wrong argument type, csv_text=42:
{
  "jsonrpc": "2.0",
  "id": 5,
  "error": {
    "code": -32602,
    "message": "Unknown tool or invalid arguments"
  }
}


## 3. Retrieve a decision in a new session

Session A chose `csv.writer` after finding the quoting bug. That choice remained
in its history. The project instructions only require interface compatibility
and regression tests. Session B starts with "Let's add another column."

**MemPalace: [official site](https://mempalaceofficial.com/) ·
[GitHub README](https://github.com/MemPalace/mempalace).**

The site presents memory as continuity between sessions. Its conversation replay
and palace diagram show the intended experience: keep original text, organize
it, then retrieve selected material when it becomes relevant.

In the README, let's inspect [What it is](https://github.com/MemPalace/mempalace#what-it-is),
[MCP server](https://github.com/MemPalace/mempalace#mcp-server) and
[Auto-save hooks](https://github.com/MemPalace/mempalace#auto-save-hooks):
verbatim storage, scoped semantic search, and integration with an agent's lifecycle.
The [Benchmarks](https://github.com/MemPalace/mempalace#benchmarks) section reports
retrieval metrics; the metric, evaluation split and use of an LLM reranker matter
when interpreting the headline. The documented CLI includes
`mempalace search "CSV quoting"`.

Below we inspect a small memory fixture using literal matching and project/status
filters. It is not a MemPalace run or a semantic-retrieval benchmark.

In [6]:
memory_records = [
    {"id": "csv-quoting", "project": "csv-export", "status": "active",
     "text": "CSV quoting: use csv.writer; joining fields breaks names containing commas.",
     "source": "Section 1: parsed fields for Doe, Jane"},
    {"id": "other-project", "project": "legacy-import", "status": "active",
     "text": "CSV quoting uses a semicolon-delimited legacy import.",
     "source": "A different project's import contract"},
    {"id": "old-choice", "project": "csv-export", "status": "superseded",
     "text": "CSV quoting: join fields with commas.", "source": "Before the regression"},
]
new_session = [{"role": "user", "content": "Let's add another column to the export."}]
hits = [record for record in memory_records
        if record["project"] == "csv-export" and record["status"] == "active"
        and "quoting" in record["text"].lower()]
assert [record["id"] for record in hits] == ["csv-quoting"]
request_with_memory = {"messages": new_session, "retrieved_memory": hits}
for title, value in [("New session", new_session), ("Retrieved record", hits),
                     ("Context packet after retrieval (provider-agnostic)", request_with_memory)]:
    print(title + ":")
    print(json.dumps(value, indent=2), "\n")

New session:
[
  {
    "role": "user",
    "content": "Let's add another column to the export."
  }
] 

Retrieved record:
[
  {
    "id": "csv-quoting",
    "project": "csv-export",
    "status": "active",
    "text": "CSV quoting: use csv.writer; joining fields breaks names containing commas.",
    "source": "Section 1: parsed fields for Doe, Jane"
  }
] 

Context packet after retrieval (provider-agnostic):
{
  "messages": [
    {
      "role": "user",
      "content": "Let's add another column to the export."
    }
  ],
  "retrieved_memory": [
    {
      "id": "csv-quoting",
      "project": "csv-export",
      "status": "active",
      "text": "CSV quoting: use csv.writer; joining fields breaks names containing commas.",
      "source": "Section 1: parsed fields for Doe, Jane"
    }
  ]
} 



The source helps us recheck an old decision. A stored record influences a
model request only after the selected text enters its context. The final packet
above illustrates context assembly; the API message format depends on the provider.

## 4. Account for prefill across tool rounds

Let's assign input lengths. The first prompt has 4,000 tokens. A model emits
200 tokens before the tool call; its tool returns 8,000. The next round adds
another 200 generated tokens and a 2,000-token result.
These are chosen lengths, not tokenization of the small CSV above.

```text
Input 1: [initial prompt]
Input 2: [input 1][generated call][tool result]
Input 3: [input 2][generated call][tool result]
```

A client-side tool loop ends the current generation, executes the tool and
submits a continuation. Parallel tool calls can share one continuation.

In [7]:
initial_prompt = 4_000
generated_before_tool = [200, 200]
tool_results = [8_000, 2_000]
prompt_lengths = [initial_prompt]
for generated, observation in zip(generated_before_tool, tool_results):
    prompt_lengths.append(prompt_lengths[-1] + generated + observation)

# Assume a hit on each preceding prompt; prior generated output is not cached here.
cached_prefix = [0] + prompt_lengths[:-1]
new_prefill = [full - cached for full, cached in zip(prompt_lengths, cached_prefix)]
rows = "".join(
    f"<tr><td>{i}</td><td>{full:,}</td><td>{cached:,}</td><td>{suffix:,}</td></tr>"
    for i, (full, cached, suffix) in enumerate(zip(prompt_lengths, cached_prefix, new_prefill), 1)
)
totals = [sum(prompt_lengths), sum(cached_prefix), sum(new_prefill)]
table = (
    '<table style="border-collapse:collapse;font-size:16px" border="1" cellpadding="10">'
    '<tr><th>Request</th><th>Total input</th><th>Cached prefix</th><th>New prefill</th></tr>'
    + rows + '<tr><th>Total</th>' + "".join(f"<th>{v:,}</th>" for v in totals) + "</tr></table>"
)
display(HTML(table))
print(f"No reuse: {totals[0]:,} input positions processed across requests.")
print(f"Stated hits: {totals[2]:,} uncached positions and {totals[1]:,} reused positions.")
assert prompt_lengths == [4000, 12200, 14400]
assert totals == [30600, 16200, 14400]

Request,Total input,Cached prefix,New prefill
1,"4,000",0,"4,000"
2,"12,200","4,000","8,200"
3,"14,400","12,200","2,200"
Total,"30,600","16,200","14,400"


No reuse: 30,600 input positions processed across requests.
Stated hits: 14,400 uncached positions and 16,200 reused positions.


In [8]:
# Draw the same quantities; SVG widths come from the computed token counts.
scale = 700 / max(prompt_lengths)
svg = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1000 230" '
       'style="max-width:1000px;font:16px sans-serif;background:white">']
svg += ['<rect x="125" y="10" width="18" height="18" fill="#90bddc"/>',
        '<text x="151" y="25">Reused prefix</text>',
        '<rect x="330" y="10" width="18" height="18" fill="#d8a16a"/>',
        '<text x="356" y="25">New prefill</text>']
for i, (full, cached, suffix) in enumerate(zip(prompt_lengths, cached_prefix, new_prefill)):
    y = 55 + i * 55
    svg += [f'<text x="15" y="{y + 25}">Request {i + 1}</text>',
            f'<rect x="125" y="{y}" width="{cached * scale}" height="35" fill="#90bddc"/>',
            f'<rect x="{125 + cached * scale}" y="{y}" width="{suffix * scale}" height="35" fill="#d8a16a"/>',
            f'<text x="{140 + full * scale}" y="{y + 25}">{full:,}</text>']
svg.append('</svg>')
display(HTML("".join(svg)))

The first long tool result requires prefill on request 2. It becomes part of
the reusable prefix on request 3. Let's change an instruction at the beginning
of the third request and compare the newly processed input.

In [9]:
miss_on_third = new_prefill[:2] + [prompt_lengths[2]]
print(f"Full hits on prior prompts: {sum(new_prefill):,} uncached positions.")
print(f"Miss on request 3: {sum(miss_on_third):,} uncached positions.")
print("This example assumes the edit occurs at the beginning, leaving no reusable prefix.")
assert sum(miss_on_third) == 26_600
print("Actual usage needs observed cache-hit counters from the provider.")

Full hits on prior prompts: 14,400 uncached positions.
Miss on request 3: 26,600 uncached positions.
This example assumes the edit occurs at the beginning, leaving no reusable prefix.
Actual usage needs observed cache-hit counters from the provider.


Cache entries need a matching prefix and must remain available under the
provider's routing, size and lifetime rules. New text still needs prefill.
Decoding still accesses the long context's KV cache.
Token counts alone do not give a FLOP, latency or price ratio.

Sources: [Claude prompt caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching),
[OpenAI prompt caching](https://developers.openai.com/api/docs/guides/prompt-caching).

A memory search chooses which information to retrieve. Prefix caching reuses
computation for a matching input sequence. Both can affect the same request.

In [10]:
import platform
try:
    import google.colab
    runtime_name = "Google Colab"
except ImportError:
    runtime_name = "Local Python"

checks = {
    "runtime": runtime_name,
    "python": platform.python_version(),
    "csv_fields_after_fix": len(good_fields),
    "mcp_tool_execution_error": received[3]["result"]["isError"],
    "mcp_fixed_name": parsed_result["first_row"]["name"],
    "mcp_invalid_argument_code": received[5]["error"]["code"],
    "selected_memory": [record["id"] for record in hits],
    "input_positions_without_reuse": sum(prompt_lengths),
    "uncached_positions_with_stated_hits": sum(new_prefill),
}
assert checks["csv_fields_after_fix"] == 2
assert checks["mcp_invalid_argument_code"] == -32602
print(json.dumps(checks, indent=2))

{
  "runtime": "Google Colab",
  "python": "3.13.18",
  "csv_fields_after_fix": 2,
  "mcp_tool_execution_error": true,
  "mcp_fixed_name": "Doe, Jane",
  "mcp_invalid_argument_code": -32602,
  "selected_memory": [
    "csv-quoting"
  ],
  "input_positions_without_reuse": 30600,
  "uncached_positions_with_stated_hits": 14400
}
